# Streamlit Deployment Flow on EC2

This notebook walks through the complete deployment architecture and shows how each piece connects.

## Architecture Overview

```
User Browser
     ↓  (HTTPS port 443 or HTTP port 80)
EC2 Public IP / Domain
     ↓
Nginx (reverse proxy — listens on port 80/443)
     ↓  (proxies to localhost:8501)
Streamlit (running on 127.0.0.1:8501)
     ↓
Python 3.11 virtual environment
     ↓
Your app code in /opt/apps/YOUR_REPO_NAME
```

## Complete Deployment Flow

In [ ]:
deployment_steps = [
    ("1",  "Login to EC2",                    "ssh -i key.pem ec2-user@YOUR_IP"),
    ("2",  "Become root",                     "sudo su -"),
    ("3",  "Check OS",                        "cat /etc/os-release"),
    ("4",  "Update + install build tools",    "yum update -y && yum install -y gcc openssl-devel ..."),
    ("5",  "Install Python 3.11",             "make altinstall (in /usr/src/Python-3.11.9)"),
    ("6",  "Create app directory",            "mkdir -p /opt/apps"),
    ("7",  "Clone repository",               "git clone https://github.com/ORG/REPO.git"),
    ("8",  "Create .env file",               "cat > .env && chmod 600 .env"),
    ("9",  "Create venv",                    "/usr/local/bin/python3.11 -m venv venv"),
    ("10", "Install requirements",           "pip install -r requirements.txt"),
    ("11", "Test manually",                  "streamlit run app.py --server.address 0.0.0.0"),
    ("12", "Create systemd service",         "nano /etc/systemd/system/streamlit-app.service"),
    ("13", "Start systemd service",          "systemctl start streamlit-app && systemctl enable"),
    ("14", "Install Nginx",                  "yum install -y nginx"),
    ("15", "Configure Nginx proxy",          "nano /etc/nginx/conf.d/streamlit.conf"),
    ("16", "Open port 80 in security group", "AWS Console → EC2 → Security Groups"),
    ("17", "Test from browser",              "http://YOUR_EC2_PUBLIC_IP"),
]

print(f"{'#':<4} {'Step':<35} {'Command/Action'}")
print("-" * 90)
for num, step, cmd in deployment_steps:
    print(f"{num:<4} {step:<35} {cmd}")

## Streamlit Run Options

In [ ]:
run_options = {
    "--server.port 8501":              "Port to run on (default 8501)",
    "--server.address 127.0.0.1":      "Bind to localhost only (use when behind Nginx)",
    "--server.address 0.0.0.0":        "Bind to all interfaces (use for direct testing)",
    "--server.headless true":          "Disable browser auto-open and email prompt",
    "--server.maxUploadSize 200":      "Max file upload size in MB",
    "--server.enableCORS false":       "Disable CORS check (sometimes needed behind proxy)",
}

print("Streamlit command-line flags:\n")
for flag, desc in run_options.items():
    print(f"  {flag}")
    print(f"    → {desc}")
    print()

print("")
print("Example — run behind Nginx:")
print("  streamlit run app.py --server.port 8501 --server.address 127.0.0.1 --server.headless true")
print()
print("Example — run for direct browser test:")
print("  streamlit run app.py --server.port 8501 --server.address 0.0.0.0")

## systemd Service File Explained

In [ ]:
service_template = """
[Unit]
Description=Streamlit App
After=network.target

[Service]
User=root
WorkingDirectory=/opt/apps/YOUR_REPO_NAME
Environment="PATH=/opt/apps/YOUR_REPO_NAME/venv/bin:/usr/local/bin:/usr/bin:/bin"
EnvironmentFile=/opt/apps/YOUR_REPO_NAME/.env
ExecStart=/opt/apps/YOUR_REPO_NAME/venv/bin/streamlit run app.py \\
    --server.port 8501 \\
    --server.address 127.0.0.1 \\
    --server.headless true
Restart=always
RestartSec=5

[Install]
WantedBy=multi-user.target
"""

service_explained = {
    "After=network.target":           "Wait for network before starting",
    "WorkingDirectory=":              "Directory where app runs (must have app.py and .env)",
    "Environment='PATH=...'":         "Includes venv/bin in PATH so correct Python is used",
    "EnvironmentFile=.env":           "Loads all KEY=VALUE pairs from .env into environment",
    "ExecStart=venv/bin/streamlit":   "Full path to streamlit in venv — no activation needed",
    "Restart=always":                 "Auto-restart if app crashes",
    "RestartSec=5":                   "Wait 5 seconds before restarting",
    "WantedBy=multi-user.target":     "Start at normal Linux multi-user boot",
}

print("Key directives explained:")
print()
for directive, meaning in service_explained.items():
    print(f"  {directive}")
    print(f"    → {meaning}")
    print()

print("Full service file:")
print(service_template)

## Nginx Config Explained

In [ ]:
nginx_config = """
server {
    listen 80;                         # Listen on port 80 (public HTTP)
    server_name _;                     # Match any hostname

    location / {
        proxy_pass http://127.0.0.1:8501;   # Forward to Streamlit
        proxy_http_version 1.1;             # Required for WebSockets

        proxy_set_header Host $host;
        proxy_set_header X-Real-IP $remote_addr;
        proxy_set_header X-Forwarded-For $proxy_add_x_forwarded_for;
        proxy_set_header X-Forwarded-Proto $scheme;

        # These two headers are REQUIRED for Streamlit WebSocket support:
        proxy_set_header Upgrade $http_upgrade;
        proxy_set_header Connection "upgrade";

        proxy_read_timeout 86400;     # 24-hour timeout for long sessions
    }
}
"""

print(nginx_config)

## Health Check Commands

In [ ]:
health_checks = [
    ("systemctl status streamlit-app",       "Is the Streamlit service running?"),
    ("systemctl status nginx",               "Is Nginx running?"),
    ("ss -tulpn | grep 8501",                "Is Streamlit listening on port 8501?"),
    ("ss -tulpn | grep :80",                 "Is Nginx listening on port 80?"),
    ("curl http://127.0.0.1:8501",           "Can we reach Streamlit internally?"),
    ("curl http://localhost",                "Can we reach Nginx on port 80?"),
    ("journalctl -u streamlit-app -n 20",   "What do the app logs say?"),
    ("tail -20 /var/log/nginx/error.log",    "What do the Nginx error logs say?"),
]

print("Run these after deployment to verify everything is working:\n")
for cmd, question in health_checks:
    print(f"$ {cmd}")
    print(f"  → {question}")
    print()

## Updating the App After Code Changes

In [ ]:
update_flow = """
# 1. Go to app directory
cd /opt/apps/YOUR_REPO_NAME

# 2. Pull latest code
git pull origin main

# 3. Activate venv and install any new packages
source venv/bin/activate
pip install -r requirements.txt
deactivate

# 4. Restart the service
systemctl restart streamlit-app

# 5. Verify
systemctl status streamlit-app
journalctl -u streamlit-app -n 20
"""

print("Update deployment flow:")
print(update_flow)

## Security Group Summary

In [ ]:
security_group_rules = [
    {"type": "Inbound",  "port": 80,   "source": "0.0.0.0/0",       "open": True,  "reason": "Nginx HTTP (public)"},
    {"type": "Inbound",  "port": 443,  "source": "0.0.0.0/0",       "open": True,  "reason": "Nginx HTTPS (public)"},
    {"type": "Inbound",  "port": 22,   "source": "YOUR_ADMIN_IP/32", "open": True,  "reason": "SSH admin only"},
    {"type": "Inbound",  "port": 8501, "source": "0.0.0.0/0",       "open": False, "reason": "❌ DO NOT OPEN — internal only"},
    {"type": "Outbound", "port": "All", "source": "0.0.0.0/0",      "open": True,  "reason": "Allow all outbound (default)"},
]

print(f"{'Direction':<10} {'Port':<8} {'Source':<22} {'Open?':<8} {'Reason'}")
print("-" * 80)
for rule in security_group_rules:
    status = "✅ YES" if rule["open"] else "❌ NO"
    print(f"{rule['type']:<10} {str(rule['port']):<8} {rule['source']:<22} {status:<8} {rule['reason']}")